In [1]:
# ============================================================
# CELL 1 — Imports
# ============================================================
import os
import numpy as np
import librosa
import pandas as pd

print("✅ Imports done")

✅ Imports done


In [2]:
# ============================================================
# CELL 2 — Paths and thresholds
# ============================================================
BASE_DIR      = "/Users/abey/Documents/pitch"
REFERENCE_DIR = os.path.join(BASE_DIR, "reference")
MODELS_DIR    = os.path.join(BASE_DIR, "models")

# pitch median delta — register check
# |ref_median - tts_median| > 30 Hz → wrong pitch zone
PITCH_MEDIAN_THRESHOLD = 30.0

# pitch std absolute floor — minimum expressiveness
# tts_std < 20 Hz → too flat regardless of reference
PITCH_STD_ABS_THRESHOLD = 20.0

# pitch std delta — relative expressiveness vs reference
# tts_std < ref_std * 0.5 → TTS is less than half as expressive as reference
# using ratio instead of absolute delta because cross-lingual
# std values are on different scales
PITCH_STD_RATIO_THRESHOLD = 0.5

print(f"Pitch Median Threshold    : ±{PITCH_MEDIAN_THRESHOLD} Hz")
print(f"Pitch Std Abs Threshold   : {PITCH_STD_ABS_THRESHOLD} Hz minimum")
print(f"Pitch Std Ratio Threshold : TTS must be ≥ {PITCH_STD_RATIO_THRESHOLD}x reference std")
print("✅ Paths and thresholds set")

Pitch Median Threshold    : ±30.0 Hz
Pitch Std Abs Threshold   : 20.0 Hz minimum
Pitch Std Ratio Threshold : TTS must be ≥ 0.5x reference std
✅ Paths and thresholds set


In [3]:
# ============================================================
# CELL 3 — compute_pitch function
# ============================================================
def compute_pitch(audio_path):
    try:
        audio, sr = librosa.load(audio_path, sr=None, mono=True)

        f0, voiced_flag, voiced_probs = librosa.pyin(
            audio,
            fmin=librosa.note_to_hz('C2'),
            fmax=librosa.note_to_hz('C7'),
            sr=sr
        )

        voiced_f0 = f0[voiced_flag]

        if len(voiced_f0) == 0:
            print(f"  ⚠️  No voiced frames: {os.path.basename(audio_path)}")
            return None, None, 0.0

        pitch_median     = round(float(np.median(voiced_f0)), 2)
        pitch_std        = round(float(np.std(voiced_f0)), 2)
        voiced_ratio     = round(float(np.sum(voiced_flag) / len(voiced_flag)), 3)

        return pitch_median, pitch_std, voiced_ratio

    except Exception as e:
        print(f"  ⚠️  Pitch error: {e}")
        return None, None, None

print("✅ compute_pitch defined")

✅ compute_pitch defined


In [4]:
# ============================================================
# CELL 4 — Startup validation
# ============================================================
if not os.path.exists(MODELS_DIR):
    raise FileNotFoundError(f"Models folder not found: {MODELS_DIR}")

reference_available = os.path.exists(REFERENCE_DIR)
if reference_available:
    ref_files = sorted([f for f in os.listdir(REFERENCE_DIR) if f.endswith(".wav")])
    print(f"✅ Reference folder found: {len(ref_files)} files")
else:
    print(f"⚠️  No reference folder — median delta and std ratio disabled")
    print(f"   Only absolute std floor check will run")

model_folders = sorted([
    d for d in os.listdir(MODELS_DIR)
    if os.path.isdir(os.path.join(MODELS_DIR, d))
])
if not model_folders:
    raise ValueError(f"No model folders found in {MODELS_DIR}")
print(f"✅ Models found: {model_folders}")

model_samples = {}
for model in model_folders:
    model_path = os.path.join(MODELS_DIR, model)
    wav_files  = sorted([
        f for f in os.listdir(model_path)
        if f.endswith(".wav")
    ])
    model_samples[model] = wav_files
    print(f"   {model}: {len(wav_files)} samples")

reference_filenames = set(model_samples[model_folders[0]])
for model in model_folders[1:]:
    current_filenames = set(model_samples[model])
    if current_filenames != reference_filenames:
        missing = reference_filenames - current_filenames
        extra   = current_filenames - reference_filenames
        raise ValueError(
            f"Model '{model}' has mismatched filenames.\n"
            f"  Missing : {missing}\n"
            f"  Extra   : {extra}"
        )
print("✅ All models have identical filenames")

sample_names = model_samples[model_folders[0]]
total        = len(model_folders) * len(sample_names)
print(f"\nReady: {len(model_folders)} models × {len(sample_names)} samples = {total} evaluations")
print(f"Reference: {'available' if reference_available else 'not available'}")

✅ Reference folder found: 2 files
✅ Models found: ['m1']
   m1: 2 samples
✅ All models have identical filenames

Ready: 1 models × 2 samples = 2 evaluations
Reference: available


In [5]:
# ============================================================
# CELL 5 — Main evaluation loop
# ============================================================
results = []

for model in model_folders:
    print(f"\n{'='*50}")
    print(f"Model: {model}")
    print(f"{'='*50}")

    for wav_file in model_samples[model]:
        sample_name = os.path.splitext(wav_file)[0]
        tts_path    = os.path.join(MODELS_DIR, model, wav_file)

        print(f"\n  Sample : {sample_name}")

        # ── compute TTS pitch ──
        tts_median, tts_std, tts_voiced_ratio = compute_pitch(tts_path)
        print(f"  TTS    : median={tts_median}Hz | std={tts_std}Hz | voiced={tts_voiced_ratio}")

        # ── defaults ──
        ref_median       = None
        ref_std          = None
        ref_voiced_ratio = None
        median_delta     = None
        std_ratio        = None
        median_pass      = None
        std_ratio_pass   = None
        ref_flag         = "NO_REF"
        is_degraded      = False

        # ── compute reference pitch if available ──
        if reference_available:
            ref_path = os.path.join(REFERENCE_DIR, wav_file)
            if os.path.exists(ref_path):
                ref_median, ref_std, ref_voiced_ratio = compute_pitch(ref_path)
                print(f"  Ref    : median={ref_median}Hz | std={ref_std}Hz | voiced={ref_voiced_ratio}")

                # degraded only if reference voiced ratio is too low to trust
                if ref_voiced_ratio is not None and ref_voiced_ratio < 0.2:
                    ref_flag    = "REF_UNVOICED"
                    is_degraded = True
                    print(f"  ⚠️  Reference voiced ratio < 0.2 — delta checks unreliable")
                else:
                    ref_flag = "—"

                    if ref_median is not None and tts_median is not None:
                        median_delta = round(abs(ref_median - tts_median), 2)
                        median_pass  = median_delta <= PITCH_MEDIAN_THRESHOLD

                    if ref_std is not None and tts_std is not None and ref_std > 0:
                        std_ratio      = round(tts_std / ref_std, 3)
                        std_ratio_pass = std_ratio >= PITCH_STD_RATIO_THRESHOLD

        # ── absolute std floor check — always runs, no reference needed ──
        if tts_std is not None:
            std_abs_pass = tts_std >= PITCH_STD_ABS_THRESHOLD
        else:
            std_abs_pass = None

        # ── final pass/fail ──
        if tts_median is None:
            final_pass = "⚠️ ERROR"
        else:
            failures = []

            if std_abs_pass is False:
                failures.append("Flat (abs)")

            if std_ratio_pass is False:
                failures.append("Flat (vs ref)")

            if median_pass is False:
                failures.append("Register")

            if not failures:
                final_pass = "✅ PASS"
            else:
                final_pass = f"❌ FAIL ({', '.join(failures)})"

        print(f"  Result : {final_pass} | Median Δ: {median_delta} | Std Ratio: {std_ratio} | Degraded: {is_degraded}")

        results.append({
            "Model"          : model,
            "Sample"         : sample_name,
            "TTS Median"     : tts_median,
            "TTS Std"        : tts_std,
            "TTS Voiced"     : tts_voiced_ratio,
            "Ref Median"     : ref_median,
            "Ref Std"        : ref_std,
            "Ref Voiced"     : ref_voiced_ratio,
            "Median Delta"   : median_delta,
            "Std Ratio"      : std_ratio,
            "Std Abs Pass"   : "✅" if std_abs_pass else "❌" if std_abs_pass is not None else "—",
            "Std Ratio Pass" : "✅" if std_ratio_pass else "❌" if std_ratio_pass is not None else "—",
            "Median Pass"    : "✅" if median_pass else "❌" if median_pass is not None else "—",
            "Final Pass"     : final_pass,
            "Ref Flag"       : ref_flag,
            "_is_degraded"   : is_degraded,
        })

print("\n\nAll evaluations complete.")




Model: m1

  Sample : YASH 2


/Users/abey/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  TTS    : median=145.15Hz | std=51.72Hz | voiced=0.657
  Ref    : median=145.99Hz | std=62.46Hz | voiced=0.609
  Result : ✅ PASS | Median Δ: 0.84 | Std Ratio: 0.828 | Degraded: False

  Sample : YASH_01
  TTS    : median=145.99Hz | std=62.46Hz | voiced=0.609
  Ref    : median=145.15Hz | std=51.72Hz | voiced=0.657
  Result : ✅ PASS | Median Δ: 0.84 | Std Ratio: 1.208 | Degraded: False


All evaluations complete.


In [6]:

# ============================================================
# CELL 6 — Results and model comparison
# ============================================================
df = pd.DataFrame(results)

# ── Table 1 — full per segment ──
print("\n========== FULL PER-SEGMENT RESULTS ==========")
print(df[[
    "Model", "Sample",
    "TTS Median", "TTS Std", "TTS Voiced",
    "Ref Median", "Ref Std", "Ref Voiced",
    "Median Delta", "Std Ratio",
    "Std Abs Pass", "Std Ratio Pass", "Median Pass",
    "Final Pass", "Ref Flag"
]].to_string(index=False))

# ── Table 2 — per model summary ──
print("\n========== MODEL COMPARISON SUMMARY ==========")
summary_rows = []

for model in model_folders:
    model_df    = df[df["Model"] == model]
    clean_df    = model_df[~model_df["_is_degraded"]]
    degraded_df = model_df[model_df["_is_degraded"]]
    total       = len(model_df)

    # clean pass rate — primary ranking number
    clean_total   = len(clean_df)
    clean_pass    = (clean_df["Final Pass"] == "✅ PASS").sum()

    # degraded pass rate — secondary
    deg_total     = len(degraded_df)
    deg_pass      = (degraded_df["Final Pass"] == "✅ PASS").sum()

    flat_abs_count   = model_df["Final Pass"].str.contains("Flat \\(abs\\)").sum()
    flat_ratio_count = model_df["Final Pass"].str.contains("Flat \\(vs ref\\)").sum()
    register_count   = model_df["Final Pass"].str.contains("Register").sum()

    mean_tts_std   = round(model_df["TTS Std"].dropna().mean(), 2)
    mean_std_ratio = round(model_df["Std Ratio"].dropna().mean(), 3) if model_df["Std Ratio"].notna().any() else None
    mean_delta     = round(model_df["Median Delta"].dropna().mean(), 2) if model_df["Median Delta"].notna().any() else None

    summary_rows.append({
        "Model"            : model,
        "Total Segments"   : total,
        "Clean Segments"   : clean_total,
        "Clean Pass Rate"  : f"{clean_pass}/{clean_total}"  if clean_total > 0 else "—",
        "Degraded Segments": deg_total,
        "Degraded Pass Rate": f"{deg_pass}/{deg_total}"     if deg_total > 0 else "—",
        "Flat Abs Fails"   : flat_abs_count,
        "Flat Ratio Fails" : flat_ratio_count,
        "Register Fails"   : register_count,
        "Mean TTS Std"     : mean_tts_std,
        "Mean Std Ratio"   : mean_std_ratio,
        "Mean Median Delta": mean_delta,
    })

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# ── Table 3 — model ranking ──
print("\n========== MODEL RANKING ==========")
print("Primary   → Clean Pass Rate (ref voiced ratio >= 0.2 segments only)")
print("Tiebreak1 → Degraded Pass Rate")
print("Tiebreak2 → Mean Std Ratio (highest first)\n")

def parse_rate(rate_str):
    if rate_str == "—":
        return -1
    return int(rate_str.split("/")[0])

summary_df["_clean_pass_num"]    = summary_df["Clean Pass Rate"].apply(parse_rate)
summary_df["_degraded_pass_num"] = summary_df["Degraded Pass Rate"].apply(parse_rate)
summary_df["_mean_std_ratio"]    = summary_df["Mean Std Ratio"].fillna(-999)

ranking = summary_df.sort_values(
    by=["_clean_pass_num", "_degraded_pass_num", "_mean_std_ratio"],
    ascending=[False, False, False]
)[[
    "Model", "Clean Pass Rate", "Degraded Pass Rate",
    "Flat Abs Fails", "Flat Ratio Fails", "Register Fails",
    "Mean TTS Std", "Mean Std Ratio", "Mean Median Delta"
]]

print(ranking.to_string(index=False))

print("\n========== WHAT TO LOOK FOR ==========")
print("Clean Pass Rate    → primary ranking — ref voiced ratio >= 0.2 segments only")
print("Degraded Pass Rate → segments where reference had too few voiced frames")
print("Flat Abs Fails     → TTS std below 20Hz floor — robotic regardless of reference")
print("Flat Ratio Fails   → TTS less than 50% as expressive as reference")
print("Register Fails     → TTS pitch zone wrong — fix voice clone")
print("Mean Std Ratio     → 1.0 = same expressiveness as reference | <0.5 = much flatter")
print("Mean Median Delta  → average pitch register difference from reference in Hz")
print(f"\nThresholds:")
print(f"  Median delta  ≤ {PITCH_MEDIAN_THRESHOLD} Hz")
print(f"  Std abs       ≥ {PITCH_STD_ABS_THRESHOLD} Hz")
print(f"  Std ratio     ≥ {PITCH_STD_RATIO_THRESHOLD}x reference")



========== FULL PER-SEGMENT RESULTS ==========
Model  Sample  TTS Median  TTS Std  TTS Voiced  Ref Median  Ref Std  Ref Voiced  Median Delta  Std Ratio Std Abs Pass Std Ratio Pass Median Pass Final Pass Ref Flag
   m1  YASH 2      145.15    51.72       0.657      145.99    62.46       0.609          0.84      0.828            ✅              ✅           ✅     ✅ PASS        —
   m1 YASH_01      145.99    62.46       0.609      145.15    51.72       0.657          0.84      1.208            ✅              ✅           ✅     ✅ PASS        —

========== MODEL COMPARISON SUMMARY ==========
Model  Total Segments  Clean Segments Clean Pass Rate  Degraded Segments Degraded Pass Rate  Flat Abs Fails  Flat Ratio Fails  Register Fails  Mean TTS Std  Mean Std Ratio  Mean Median Delta
   m1               2               2             2/2                  0                  —               0                 0               0         57.09           1.018               0.84

========== MODEL RANKING =

In [7]:
# final cell in each gate notebook
df.to_csv(os.path.join(BASE_DIR, "results.csv"), index=False)
print("✅ Results saved to results.csv")

✅ Results saved to results.csv
